In [1]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import re
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# user-defined paths
base_dir = '../../../../datos-notebooks/3.global-data-cba/'
ground_data_name = 'ground_data_170226_validation.gpkg'

In [3]:
# derived paths
ground_data_path = os.path.join(base_dir, 'MRT-prediction/ground_data', ground_data_name)
mrt_path = os.path.join(base_dir, 'MRT-prediction/MRT_1m-15dif.tif')

#### reading and fixing ground data 

In [4]:
gdf = gpd.read_file(ground_data_path)
len(gdf)


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column start, 2026-02-17T13:38:25.282-03:00, successfully parsed
  return ogr_read(


26

In [5]:
# correct TG1
# Create corrected TG column based on instrument 
gdf["tg_corr"] = np.where(
    gdf["Instrument"] == "T1 - Sper Scientific 800036 ",
    0.639 * gdf["TG"] + 9.681,   # apply correction for old instrument from notebook 3.4.4.1.
    gdf["TG"]                   # keep original value for new instrument
)


In [6]:
gdf.head(3)

,start,end,today,operador,Instrument,condition,TA,TG,WIND M/S,_Location_latitude,_Location_longitude,_Location_altitude,_Location_precision,picture 1,picture 1_URL,picture 2,picture 2_URL,context,geometry,tg_corr
0,2026-02-17 13:38:25.282000-03:00,2026-02-17 13:40:22.997000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SHADOW,28.7,31.6,2.7,-31.370683,-64.248834,418.516928,4.985490,image-13_40_8.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,image-13_40_19.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (-64.24834 -31.36998),31.6000
1,2026-02-17 13:18:53.548000-03:00,2026-02-17 13:41:19.664000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SHADOW,29.3,32.6,0.0,-31.370702,-64.248856,436.989990,37.895014,image-13_40_38.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,image-13_40_47.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (-64.24836 -31.36995),30.5124
2,2026-02-17 13:40:23.037000-03:00,2026-02-17 13:44:25.264000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SUN,30.1,38.7,1.6,-31.370693,-64.248813,419.412069,6.055630,image-13_44_10.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,image-13_44_18.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (-64.24881 -31.37069),38.7000


correcting wind

In [7]:
# Subtract 0.88 for wind extracted with anemometer 2 (grande)
gdf["WIND M/S corr"] = np.where(gdf["Instrument"] == "T2 - TM-188D TENMARS ", (gdf["WIND M/S"] - 1.0102) / 0.8785, gdf["WIND M/S"])

gdf["WIND M/S corr"] = np.where(gdf["WIND M/S corr"] <= 0, 0.01, gdf["WIND M/S corr"]) # set minimum 0.01 to avoid issues


In [8]:
# calculate mean of wind values
# Convert 'start' to datetime
gdf["end"] = pd.to_datetime(gdf["start"], errors="coerce")

# Sort by time
gdf = gdf.sort_values("start")

# Set as index
gdf = gdf.set_index("start")

# Compute centered rolling mean over 10 minutes (±5 min)
gdf["wind_mean"] = (
    gdf["WIND M/S corr"]
    .rolling(window="10min", center=True, min_periods=1)
    .mean()
    .round(2)
)

# Reset index if necesitas la columna 'start' de nuevo
gdf = gdf.reset_index()

In [9]:
# Parameters
eps = 0.95
D1 = 0.04   # diameter of globe 1 [m] 
D2 = 0.05  # diameter of globe 2 [m] 

# Select diameter according to instrument
D = np.where(gdf["Instrument"] == "T1 - Sper Scientific 800036 ", D1,D2)

# MRT calculation using row-specific diameter
# wind used 0.1 for all cases because values were introducing errores and the changes are not significant (all under 1 m/s)
gdf["MRT"] = (
    (
        (gdf["tg_corr"] + 273.15)**4
        + (1.1e8 * gdf["wind_mean"]**0.6 / (eps * D**0.4)) * (gdf["tg_corr"] - gdf["TA"]) 
    )**0.25
    - 273.15
)

Probando con MRT 

In [10]:
# read raster crs and sample values
with rasterio.open(mrt_path) as src:
    mrt_crs = src.crs
    points_proj = gdf.to_crs(mrt_crs)
    samples = list(src.sample(zip(points_proj.geometry.x, points_proj.geometry.y)))
    points_proj['MRT_predicted'] = [s[0] for s in samples]

points_proj['MRT_predicted'] = points_proj['MRT_predicted'].astype(float) 

# METRICS
obs = points_proj['MRT']
pred = points_proj['MRT_predicted']

points_proj['residual'] = pred - obs

# export gpkg
out_gpkg = os.path.join(base_dir, 'MRT-prediction/MRT_points_2026.gpkg')
points_proj.to_file(out_gpkg, driver="GPKG")

In [11]:
points_proj.head()

,start,end,today,operador,Instrument,condition,TA,TG,WIND M/S,_Location_latitude,...,picture 2,picture 2_URL,context,geometry,tg_corr,WIND M/S corr,wind_mean,MRT,MRT_predicted,residual
0,2026-02-17 13:18:53.548000-03:00,2026-02-17 13:18:53.548000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SHADOW,29.3,32.6,0.0,-31.370702,...,image-13_40_47.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381282.355 6528723.634),30.5124,0.010000,0.01,30.798583,36.787640,5.989056
1,2026-02-17 13:38:25.282000-03:00,2026-02-17 13:38:25.282000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SHADOW,28.7,31.6,2.7,-31.370683,...,image-13_40_19.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381284.173 6528720.501),31.6000,1.923506,1.10,41.514903,36.812473,-4.702430
2,2026-02-17 13:40:23.037000-03:00,2026-02-17 13:40:23.037000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SUN,30.1,38.7,1.6,-31.370693,...,image-13_44_18.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381240.071 6528640.976),38.7000,0.671372,0.83,60.610154,52.308060,-8.302094
3,2026-02-17 13:41:19.707000-03:00,2026-02-17 13:41:19.707000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SUN,28.9,42.2,0.7,-31.370663,...,image-13_44_45.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381237.656 6528644.274),36.6468,0.700000,0.70,56.684216,52.330524,-4.353692
4,2026-02-17 13:44:25.303000-03:00,2026-02-17 13:44:25.303000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SHADOW,30.7,34.4,0.9,-31.361940,...,image-13_58_2.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,residential,POINT (382446.075 6529619.412),34.4000,0.010000,0.40,41.212535,31.941757,-9.270778


In [12]:

mae = np.mean(np.abs(obs - pred))
rmse = np.sqrt(np.mean((obs - pred)**2))
bias = np.mean(pred - obs)

print("MAE:", mae)
print("RMSE:", rmse)
print("Bias:", bias)

MAE: 5.683978086624135
RMSE: 6.742696290930003
Bias: -0.5473483853071326


In [13]:
import plotly.express as px
import numpy as np
import pandas as pd

obs = points_proj['MRT']
pred = points_proj['MRT_predicted']

# Compute 1:1 line range
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

# Scatter
fig = px.scatter(
    points_proj,
    x=points_proj['MRT'],
    y=points_proj['MRT_predicted'],
    color="condition",
    hover_data=["TG", "TA","WIND M/S", "Instrument"],
)

# 1:1 line
fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="points_proj comparison"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()


In [14]:
# Compute 1:1 line range
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

# Scatter
fig = px.scatter(
    points_proj,
    x=obs,
    y=pred,
    color="Instrument",
    hover_data=["TG", "TA","WIND M/S", "condition"],
)

# 1:1 line
fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="points_proj comparison"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [15]:
# Compute 1:1 line range
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

# Scatter
fig = px.scatter(
    points_proj,
    x=obs,
    y=pred,
    color="context",
    hover_data=["TG", "TA","WIND M/S", "condition"],
)

# 1:1 line
fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="points_proj comparison"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [16]:
residuals = obs - pred  # Compute residuals
# Scatter
fig = px.scatter(
    points_proj,
    x=pred,
    y=residuals,
    color="context",
    hover_data=["TG", "TA","WIND M/S corr", "condition", "Instrument"],
)

# 1:1 line
fig.add_scatter(
    x=[pred.min(), pred.max()],
    y=[0, 0],
    mode="lines",
    name="Zero residual line"
)

fig.update_layout(
    xaxis_title="Predicted MRT °C",
    yaxis_title="Residuals (Obs - Pred)",
    title="points_proj residuals"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [17]:
points_proj

,start,end,today,operador,Instrument,condition,TA,TG,WIND M/S,_Location_latitude,...,picture 2,picture 2_URL,context,geometry,tg_corr,WIND M/S corr,wind_mean,MRT,MRT_predicted,residual
0,2026-02-17 13:18:53.548000-03:00,2026-02-17 13:18:53.548000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SHADOW,29.3,32.6,0.0,-31.370702,...,image-13_40_47.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381282.355 6528723.634),30.5124,0.010000,0.01,30.798583,36.787640,5.989056
1,2026-02-17 13:38:25.282000-03:00,2026-02-17 13:38:25.282000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SHADOW,28.7,31.6,2.7,-31.370683,...,image-13_40_19.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381284.173 6528720.501),31.6000,1.923506,1.10,41.514903,36.812473,-4.702430
2,2026-02-17 13:40:23.037000-03:00,2026-02-17 13:40:23.037000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SUN,30.1,38.7,1.6,-31.370693,...,image-13_44_18.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381240.071 6528640.976),38.7000,0.671372,0.83,60.610154,52.308060,-8.302094
3,2026-02-17 13:41:19.707000-03:00,2026-02-17 13:41:19.707000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SUN,28.9,42.2,0.7,-31.370663,...,image-13_44_45.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (381237.656 6528644.274),36.6468,0.700000,0.70,56.684216,52.330524,-4.353692
4,2026-02-17 13:44:25.303000-03:00,2026-02-17 13:44:25.303000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SHADOW,30.7,34.4,0.9,-31.361940,...,image-13_58_2.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,residential,POINT (382446.075 6529619.412),34.4000,0.010000,0.40,41.212535,31.941757,-9.270778
5,2026-02-17 13:45:54.280000-03:00,2026-02-17 13:45:54.280000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SUN,34.4,38.0,0.2,-31.366038,...,image-13_55_31.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,park,POINT (381888.001 6529164.351),33.9630,0.200000,0.30,33.191597,45.099792,11.908195
6,2026-02-17 13:55:34.807000-03:00,2026-02-17 13:55:34.807000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SUN,35.0,40.9,0.8,-31.366234,...,image-13_57_43.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,park,POINT (381871.888 6529142.441),35.8161,0.800000,0.59,37.909708,45.337505,7.427797
7,2026-02-17 13:57:46.678000-03:00,2026-02-17 13:57:46.678000-03:00,2026-02-16,Macarena,T1 - Sper Scientific 800036,SUN,36.4,42.7,0.3,-31.366936,...,image-14_0_15.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,park,POINT (381845.755 6529064.324),36.9663,0.300000,0.45,38.192638,45.798218,7.605580
8,2026-02-17 13:58:05.712000-03:00,2026-02-17 13:58:05.712000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SUN,30.2,38.6,1.6,-31.361905,...,image-13_59_48.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,residential,POINT (382435.92 6529628.692),38.6000,0.671372,0.45,53.909283,47.010258,-6.899025
9,2026-02-17 14:00:37.766000-03:00,2026-02-17 14:00:37.766000-03:00,2026-02-16,Guadalupe,T2 - TM-188D TENMARS,SHADOW,28.9,33.0,1.0,-31.366773,...,image-14_9_39.jpg,https://kf.kobotoolbox.org/api/v2/assets/aF3fL...,open parking,POINT (383935.746 6529136.461),33.0000,0.010000,0.33,39.817595,38.522236,-1.295359


AJUSTANDO MRT PRED

In [18]:
mrt_path = os.path.join(base_dir, 'MRT-prediction/MRT_1m-15dif-0.8dif.tif')

In [19]:
# read raster crs and sample values
with rasterio.open(mrt_path) as src:
    mrt_crs = src.crs
    points_proj = gdf.to_crs(mrt_crs)
    samples = list(src.sample(zip(points_proj.geometry.x, points_proj.geometry.y)))
    points_proj['MRT_predicted'] = [s[0] for s in samples]

points_proj['MRT_predicted'] = points_proj['MRT_predicted'].astype(float) 

# METRICS
obs = points_proj['MRT']
pred = points_proj['MRT_predicted']

points_proj['residual'] = pred - obs


In [20]:
mae = np.mean(np.abs(obs - pred))
rmse = np.sqrt(np.mean((obs - pred)**2))
bias = np.mean(pred - obs)

print("MAE:", mae)
print("RMSE:", rmse)
print("Bias:", bias)

MAE: 5.6292493118226705
RMSE: 6.717949058472688
Bias: 0.15529273501663546


In [21]:
# Compute 1:1 line range
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

# Scatter
fig = px.scatter(
    points_proj,
    x=obs,
    y=pred,
    color="condition",
    hover_data=["TG", "TA","WIND M/S", "Instrument"],
)

# 1:1 line
fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="points_proj comparison"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [22]:
# Compute 1:1 line range
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

# Scatter
fig = px.scatter(
    points_proj,
    x=obs,
    y=pred,
    color="Instrument",
    hover_data=["TG", "TA","WIND M/S", "condition"],
)

# 1:1 line
fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="points_proj comparison"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [23]:
# Compute 1:1 line range
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

# Scatter
fig = px.scatter(
    points_proj,
    x=obs,
    y=pred,
    color="context",
    hover_data=["TG", "TA","WIND M/S", "condition"],
)

# 1:1 line
fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="points_proj comparison"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [24]:
residuals = obs - pred  # Compute residuals
# Scatter
fig = px.scatter(
    points_proj,
    x=pred,
    y=residuals,
    color="context",
    hover_data=["TG", "TA","WIND M/S", "condition", "Instrument"],
)

# 1:1 line
fig.add_scatter(
    x=[pred.min(), pred.max()],
    y=[0, 0],
    mode="lines",
    name="Zero residual line"
)

fig.update_layout(
    xaxis_title="Predicted MRT °C",
    yaxis_title="Residuals (Obs - Pred)",
    title="points_proj residuals"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [25]:
# de acá podemos entender las limitaciones del modelo: 
# no considera reflexión de objetos cercanos, 
# por lo que en residencial (donde las casas cerca reflejan) la observada fue mayor que la predicha
# caso contrario en parque y estacionamiento abierto, donde no hay objetos cercanos que reflejen, entonces la observada fue menor que la predicha